In [1]:
# Feature Engineering & Datenaufbereitung
# Projekt: Car Price Prediction

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import joblib


In [2]:
# 1. Daten laden

df = pd.read_csv("../data/processed/cleaned_data.csv")

df.head()

,Datum,Marke,Modell,Preis_Euro,Verkaufszahl,Kraftstoff,Getriebe,Hubraum_L,Bundesland,Kundenzufriedenheit,Jahr,Monat,Wochentag
0,2024-01-01,Mercedes-Benz,C-Klasse,66835,2,Elektro,Automatik,0.0,Berlin,4.7,2024,1,Monday
1,2024-01-01,Mercedes-Benz,E-Klasse,93803,2,Benzin,Manuell,1.2,Nrw,3.2,2024,1,Monday
2,2024-01-07,Volkswagen,Passat,45929,6,Hybrid,Manuell,2.0,Baden-Württemberg,3.2,2024,1,Sunday
3,2024-01-07,Mercedes-Benz,C-Klasse,76943,3,Diesel,Automatik,4.0,Berlin,3.4,2024,1,Sunday
4,2024-01-08,Bmw,5Er,107912,1,Elektro,Automatik,0.0,Berlin,3.2,2024,1,Monday


In [3]:
# 2. Überblick über den Datensatz

print("Shape:", df.shape)
print("\nSpalten:")
print(df.columns.tolist())

df.info()

Shape: (1200, 13)

Spalten:
['Datum', 'Marke', 'Modell', 'Preis_Euro', 'Verkaufszahl', 'Kraftstoff', 'Getriebe', 'Hubraum_L', 'Bundesland', 'Kundenzufriedenheit', 'Jahr', 'Monat', 'Wochentag']
<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Datum                1200 non-null   str    
 1   Marke                1200 non-null   str    
 2   Modell               1200 non-null   str    
 3   Preis_Euro           1200 non-null   int64  
 4   Verkaufszahl         1200 non-null   int64  
 5   Kraftstoff           1200 non-null   str    
 6   Getriebe             1200 non-null   str    
 7   Hubraum_L            1200 non-null   float64
 8   Bundesland           1200 non-null   str    
 9   Kundenzufriedenheit  1200 non-null   float64
 10  Jahr                 1200 non-null   int64  
 11  Monat                1200 non-null   int64  
 12  Wochen

In [4]:
# 3. Prüfen, ob fehlende Werte vorhanden sind

df.isnull().sum()

Datum                  0
Marke                  0
Modell                 0
Preis_Euro             0
Verkaufszahl           0
Kraftstoff             0
Getriebe               0
Hubraum_L              0
Bundesland             0
Kundenzufriedenheit    0
Jahr                   0
Monat                  0
Wochentag              0
dtype: int64

In [5]:
# 4. Fahrzeugalter prüfen

# Hinweis:
# Im Datensatz gibt es keine Spalte wie "Baujahr" oder "Erstzulassung".
# Deshalb kann das Fahrzeugalter nicht berechnet werden.
# Stattdessen wurden bereits zeitbezogene Features aus dem Datum erstellt:
# Jahr, Monat und Wochentag.

print(df[["Datum", "Jahr", "Monat", "Wochentag"]].head())

        Datum  Jahr  Monat Wochentag
0  2024-01-01  2024      1    Monday
1  2024-01-01  2024      1    Monday
2  2024-01-07  2024      1    Sunday
3  2024-01-07  2024      1    Sunday
4  2024-01-08  2024      1    Monday


In [6]:
# 5. Zielvariable definieren

target = "Preis_Euro"

y = df[target]

print(y.head())

0     66835
1     93803
2     45929
3     76943
4    107912
Name: Preis_Euro, dtype: int64


In [7]:
# 6. Relevante Features auswählen

features = [
    "Marke",
    "Modell",
    "Verkaufszahl",
    "Kraftstoff",
    "Getriebe",
    "Hubraum_L",
    "Bundesland",
    "Kundenzufriedenheit",
    "Jahr",
    "Monat",
    "Wochentag"
]

X = df[features]

X.head()

,Marke,Modell,Verkaufszahl,Kraftstoff,Getriebe,Hubraum_L,Bundesland,Kundenzufriedenheit,Jahr,Monat,Wochentag
0,Mercedes-Benz,C-Klasse,2,Elektro,Automatik,0.0,Berlin,4.7,2024,1,Monday
1,Mercedes-Benz,E-Klasse,2,Benzin,Manuell,1.2,Nrw,3.2,2024,1,Monday
2,Volkswagen,Passat,6,Hybrid,Manuell,2.0,Baden-Württemberg,3.2,2024,1,Sunday
3,Mercedes-Benz,C-Klasse,3,Diesel,Automatik,4.0,Berlin,3.4,2024,1,Sunday
4,Bmw,5Er,1,Elektro,Automatik,0.0,Berlin,3.2,2024,1,Monday


In [8]:
# 7. Kategoriale und numerische Features trennen

categorical_features = [
    "Marke",
    "Modell",
    "Kraftstoff",
    "Getriebe",
    "Bundesland",
    "Wochentag"
]

numeric_features = [
    "Verkaufszahl",
    "Hubraum_L",
    "Kundenzufriedenheit",
    "Jahr",
    "Monat"
]

print("Kategoriale Features:", categorical_features)
print("Numerische Features:", numeric_features)

Kategoriale Features: ['Marke', 'Modell', 'Kraftstoff', 'Getriebe', 'Bundesland', 'Wochentag']
Numerische Features: ['Verkaufszahl', 'Hubraum_L', 'Kundenzufriedenheit', 'Jahr', 'Monat']


In [9]:
# 8. One-Hot-Encoding für kategoriale Variablen

X_encoded = pd.get_dummies(
    X,
    columns=categorical_features,
    drop_first=True
)

X_encoded.head()

,Verkaufszahl,Hubraum_L,Kundenzufriedenheit,Jahr,Monat,Marke_Bmw,Marke_Mercedes-Benz,Marke_Opel,Marke_Volkswagen,Modell_5Er,...,Bundesland_Berlin,Bundesland_Hamburg,Bundesland_Hessen,Bundesland_Nrw,Wochentag_Monday,Wochentag_Saturday,Wochentag_Sunday,Wochentag_Thursday,Wochentag_Tuesday,Wochentag_Wednesday
0,2,0.0,4.7,2024,1,False,True,False,False,False,...,True,False,False,False,True,False,False,False,False,False
1,2,1.2,3.2,2024,1,False,True,False,False,False,...,False,False,False,True,True,False,False,False,False,False
2,6,2.0,3.2,2024,1,False,False,False,True,False,...,False,False,False,False,False,False,True,False,False,False
3,3,4.0,3.4,2024,1,False,True,False,False,False,...,True,False,False,False,False,False,True,False,False,False
4,1,0.0,3.2,2024,1,True,False,False,False,True,...,True,False,False,False,True,False,False,False,False,False


In [10]:
# 9. Neue Form nach One-Hot-Encoding prüfen

print("Originale Feature-Anzahl:", X.shape[1])
print("Feature-Anzahl nach Encoding:", X_encoded.shape[1])
print("Shape:", X_encoded.shape)

Originale Feature-Anzahl: 11
Feature-Anzahl nach Encoding: 43
Shape: (1200, 43)


In [11]:
# 10. Train-Test-Split erstellen

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (960, 43)
X_test: (240, 43)
y_train: (960,)
y_test: (240,)


In [12]:
# 11. Numerische Variablen skalieren

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test_scaled[numeric_features] = scaler.transform(X_test[numeric_features])

X_train_scaled.head()

,Verkaufszahl,Hubraum_L,Kundenzufriedenheit,Jahr,Monat,Marke_Bmw,Marke_Mercedes-Benz,Marke_Opel,Marke_Volkswagen,Modell_5Er,...,Bundesland_Berlin,Bundesland_Hamburg,Bundesland_Hessen,Bundesland_Nrw,Wochentag_Monday,Wochentag_Saturday,Wochentag_Sunday,Wochentag_Thursday,Wochentag_Tuesday,Wochentag_Wednesday
331,-0.782979,1.103407,-0.542494,-1.098588,0.423058,False,False,False,False,False,...,False,True,False,False,False,False,False,True,False,False
409,-1.572539,-1.056313,0.490145,-1.098588,0.718429,True,False,False,False,False,...,True,False,False,False,False,False,False,False,True,False
76,-1.177759,0.095538,-0.714601,-1.098588,-1.053799,True,False,False,False,True,...,False,False,False,False,False,False,True,False,False,False
868,-1.572539,1.103407,1.350678,0.910259,0.127687,True,False,False,False,False,...,False,False,False,False,True,False,False,False,False,False
138,1.585698,0.095538,-1.403027,-1.098588,-0.758427,True,False,False,False,False,...,True,False,False,False,False,True,False,False,False,False


In [13]:
# 12. Skalierte Daten prüfen

X_train_scaled[numeric_features].describe()

,Verkaufszahl,Hubraum_L,Kundenzufriedenheit,Jahr,Monat
count,960.000000,9.600000e+02,9.600000e+02,9.600000e+02,9.600000e+02
mean,0.000000,2.220446e-17,-6.587323e-16,5.551115e-17,9.621933e-17
std,1.000521,1.000521e+00,1.000521e+00,1.000521e+00,1.000521e+00
min,-1.572539,-1.056313e+00,-1.747240e+00,-1.098588e+00,-1.644541e+00
25%,-0.782979,-1.056313e+00,-8.867074e-01,-1.098588e+00,-7.584273e-01
50%,0.006580,9.553759e-02,-2.617454e-02,9.102590e-01,1.276865e-01
75%,0.796139,3.835002e-01,8.343583e-01,9.102590e-01,1.013800e+00
max,1.585698,1.823313e+00,1.694891e+00,9.102590e-01,1.604543e+00


In [15]:
# 13. Vorbereitete Daten speichern

X_train_scaled.to_csv("../data/train/X_train.csv", index=False)
X_test_scaled.to_csv("../data/train/X_test.csv", index=False)
y_train.to_csv("../data/train/y_train.csv", index=False)
y_test.to_csv("../data/train/y_test.csv", index=False)

joblib.dump(scaler, "../data/scaler.pkl")

print("Feature Engineering abgeschlossen.")

Feature Engineering abgeschlossen.


In [ ]:
# 14. Zusammenfassung

print("Zusammenfassung Feature Engineering:")
print("- Relevante Features wurden ausgewählt.")
print("- Kategoriale Variablen wurden mit One-Hot-Encoding umgewandelt.")
print("- Train-Test-Split wurde erstellt.")
print("- Numerische Variablen wurden skaliert.")
print("- Die vorbereiteten Daten wurden gespeichert.")

Zusammenfassung Feature Engineering:
- Relevante Features wurden ausgewählt.
- Kategoriale Variablen wurden mit One-Hot-Encoding umgewandelt.
- Train-Test-Split wurde erstellt.
- Numerische Variablen wurden skaliert.
- Die vorbereiteten Daten wurden gespeichert.
